In [35]:
import pandas as pd

link_stream = pd.read_csv('data/sync_net.csv', delimiter=',', names=['source', 'destination', 'timestamp'], index_col=False,skiprows=1)

/var/folders/g1/wg9xf7ks75b49ljdm6l5zy280000gp/T/ipykernel_74784/3028881048.py:3: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  link_stream = pd.read_csv('data/sync_net.csv', delimiter=',', names=['source', 'destination', 'timestamp'], index_col=False,skiprows=1)


In [37]:
nodes = pd.concat([link_stream['source'], link_stream['destination']]).unique()
print(len(nodes), "nodes in the link stream")

20 nodes in the link stream


In [38]:
node2id = {node: idx for idx, node in enumerate(nodes)}

link_stream['source'] = link_stream['source'].map(node2id)
link_stream['destination'] = link_stream['destination'].map(node2id)
link_stream['idx'] = range(len(link_stream))

In [40]:
link_stream.to_csv('data/sync_net_temp.csv', index=False)

In [1]:
import pandas as pd

link_stream = pd.read_csv('data/sync_net.csv')

In [16]:
(link_stream.head(5))

source  destination  timestamp  idx
0 10       0          185          1    0
  12       1          185          1    1
  18       2          185          1    2
  8        3          185          1    3
  5        4          185          1    4

In [471]:
import torch
import pandas as pd
import numpy as np
from typing import Dict, Tuple, Optional


class OfflineStateManager:
    """
    根据讨论版定义：

    - A_curr[u,c]：当前轮（build-null pass）从 0 开始累积的节点-社区累计量（通常 += delta_t * p_{u,c}）
    - S_curr[c]  ：在整轮结束后一次性由 A_curr 计算：
                  S_curr[c] = sum_u k_u * sqrt(A_curr[u,c])
                  (根号里不除 Tu)

    - A_old/S_old：上一轮 build-null 产出的冻结统计量，用于下一轮训练的 null loss（固定一整轮，避免目标漂移）

    其他状态：
    - p_prev[u,c]：节点上一次的 p，用于你 loss 里 CSC/平滑/增量等逻辑
    - node_time_pos + node_time_indptr/values：用于按出现序列给每次出现算 delta_t
    """

    def __init__(self, num_nodes: int, num_communities: int, device: str = "cpu"):
        self.num_nodes = num_nodes
        self.num_communities = num_communities
        self.device = device

        # -------- static stats (precomputed once) --------
        self.global_degree = torch.zeros(num_nodes, device=device, dtype=torch.float32)  # k_u
        self.node_lifespans = torch.ones(num_nodes, device=device, dtype=torch.float32)  # 可选（目前不用于 S）
        self.total_duration = 1.0
        self.m = 0.0

        # -------- time index for delta_t --------
        self.node_time_pos = torch.zeros(num_nodes, dtype=torch.long, device=device)
        self.node_time_indptr: Optional[torch.Tensor] = None  # [N+1]
        self.node_time_values: Optional[torch.Tensor] = None  # [total_appearances]

        # -------- dynamic buffers (curr / old) --------
        self.A_curr: Optional[torch.Tensor] = None  # [N,K] build-null 当前轮累积（从 0 开始）
        self.S_curr: Optional[torch.Tensor] = None  # [K]   当前轮结束后一次性计算

        self.A_old: Optional[torch.Tensor] = None   # [N,K] 训练时使用的冻结 null 统计
        self.S_old: Optional[torch.Tensor] = None   # [K]

        self.p_src_old_all: Optional[torch.Tensor] = None  # [num_instances, K] (CPU recommended)
        self.p_dst_old_all: Optional[torch.Tensor] = None  # [num_instances, K]
        self.num_instances_cached: int = 0
        # -------- other dynamic state --------
        self.p_prev: Optional[torch.Tensor] = None  # [N,K]

        # 数值稳定用的小 eps
        self.eps = 1e-12

    # ---------------------- one-time preprocessing ----------------------
    def pre_scan_data(self, link_stream: pd.DataFrame):
        """
        link_stream columns: ['source','destination','timestamp'] (timestamp int/float ok)

        预计算：
        - m, global_degree
        - total_duration
        - node_lifespans (可选)
        - node_time_indptr/node_time_values（用于 delta_t）
        """
        self.m = float(len(link_stream))
        print(f"Total links: {int(self.m)}")

        # ---- global_degree k_u ----
        all_nodes = pd.concat([link_stream["source"], link_stream["destination"]], ignore_index=True)
        all_nodes_t = torch.tensor(all_nodes.values, dtype=torch.long, device=self.device)
        self.global_degree = torch.bincount(all_nodes_t, minlength=self.num_nodes).float()

        # ---- total duration ----
        t_max = link_stream["timestamp"].max()
        t_min = link_stream["timestamp"].min()
        total_duration = t_max - t_min
        self.total_duration = float(total_duration) if total_duration > 0 else 1.0
        print(f"T_duration = {total_duration}")

        # ---- node lifespans (optional) ----
        sources = link_stream[["source", "timestamp"]].rename(columns={"source": "node"})
        destinations = link_stream[["destination", "timestamp"]].rename(columns={"destination": "node"})
        all_events = pd.concat([sources, destinations], ignore_index=True)

        node_stats = all_events.groupby("node")["timestamp"].agg(["min", "max"])
        epsilon = 1.0
        lifespans = (node_stats["max"] - node_stats["min"]).clip(lower=epsilon)

        self.node_lifespans.fill_(epsilon)
        active_ids = torch.tensor(node_stats.index.values, device=self.device, dtype=torch.long)
        active_vals = torch.tensor(lifespans.values, device=self.device, dtype=torch.float32)
        self.node_lifespans[active_ids] = active_vals
        print(f"Max Lifespan: {self.node_lifespans.max().item()}")

        # ---- build node_time_values: timestamps sorted by (node, timestamp) ----
        all_events_sorted = all_events.sort_values(["node", "timestamp"], kind="mergesort")
        nodes_np = all_events_sorted["node"].to_numpy(dtype=np.int64, copy=False)
        times_np = all_events_sorted["timestamp"].to_numpy(copy=False)

        if np.issubdtype(times_np.dtype, np.integer):
            if times_np.min() >= np.iinfo(np.int32).min and times_np.max() <= np.iinfo(np.int32).max:
                times_np = times_np.astype(np.int32, copy=False)
            else:
                times_np = times_np.astype(np.int64, copy=False)
        else:
            times_np = times_np.astype(np.float32, copy=False)

        counts = np.bincount(nodes_np, minlength=self.num_nodes).astype(np.int64)
        indptr = np.empty(self.num_nodes + 1, dtype=np.int64)
        indptr[0] = 0
        np.cumsum(counts, out=indptr[1:])

        self.node_time_indptr = torch.from_numpy(indptr).to(self.device)
        self.node_time_values = torch.from_numpy(times_np).to(self.device)

        print(f"Total node appearances stored: {self.node_time_values.numel()} (expected ~ {2*int(self.m)})")

        # epoch0 init
        self.reset_time_pos()
        self.reset_p_prev_uniform()
        self.init_old_uniform_prior()  # 给训练一个稳定的初始 old（可按需改成全 0）

    # ---------------------- epoch / pass control ----------------------
    def reset_time_pos(self):
        """Reset per-node appearance pointer (for delta_t)."""
        self.node_time_pos.zero_()

    def reset_p_prev_uniform(self):
        """Reset p_prev to uniform 1/K."""
        N, K = self.num_nodes, self.num_communities
        self.p_prev = torch.full((N, K), 1.0 / float(K), device=self.device, dtype=torch.float32)

    def reset_curr_from_zero(self):
        """
        Start a new build-null pass:
          A_curr = 0, S_curr = 0
        """
        N, K = self.num_nodes, self.num_communities
        self.A_curr = torch.zeros((N, K), device=self.device, dtype=torch.float32)
        self.S_curr = torch.zeros((K,), device=self.device, dtype=torch.float32)

    def init_old_uniform_prior(self):
        """
        Epoch0 fallback for A_old/S_old.

        说明：
        - 如果你训练第一轮 null term 希望是“非零稳定先验”，用 uniform prior 比全 0 更稳。
        - 若你坚持第一轮就是全 0，也可把 base 改成 0。
        """
        N, K = self.num_nodes, self.num_communities
        Tu = self.node_lifespans.clamp_min(1.0)
        self.A_old = (Tu[:, None] / float(K)).expand(N,K).to(device=self.device, dtype=torch.float32).clone()
        self.S_old = (self.global_degree[:, None] * torch.sqrt(self.A_old)).sum(dim=0)

    @torch.no_grad()
    def finalize_curr(self):
        """
        Compute S_curr from scratch AFTER the build-null pass:
          S_curr[c] = sum_u k_u * sqrt(A_curr[u,c])
        """
        if self.A_curr is None:
            raise RuntimeError("A_curr is None. Call reset_curr_from_zero() before accumulating.")
        self.S_curr = (self.global_degree[:, None] * torch.sqrt(self.A_curr.clamp_min(self.eps))).sum(dim=0)


    @torch.no_grad()
    def promote_curr_to_old(self, copy_A: bool = True):
        """
        Replace old buffers with current buffers (for next epoch training).

        copy_A=True：A_old 也替换（如果你的 null loss 需要 A_old）
        copy_A=False：只替换 S_old（如果 null loss 只需要 S）
        """
        if self.S_curr is None:
            raise RuntimeError("S_curr is None. Call finalize_curr() before promote_curr_to_old().")
        if copy_A:
            if self.A_curr is None:
                raise RuntimeError("A_curr is None. Cannot promote A.")
            self.A_old = self.A_curr.clone()
        self.S_old = self.S_curr.clone()

    # ---------------------- delta_t computation ----------------------
    def get_delta_for_batch(
        self,
        src: torch.Tensor,  # [B] long
        dst: torch.Tensor,  # [B] long
    ) -> Tuple[torch.Tensor, torch.Tensor, Dict[int, int]]:
        """
        Compute delta_src/delta_dst for each occurrence in this batch.
        Uses node_time_pos + temp_pos to handle repeated nodes within batch.

        Returns:
          delta_src, delta_dst: [B] float32
          occ_counts: {u: count_in_batch} for advancing node_time_pos
        """
        assert self.node_time_indptr is not None and self.node_time_values is not None, \
            "Call pre_scan_data() first to build node_time_indptr/node_time_values."

        assert src.dtype == torch.long and dst.dtype == torch.long
        B = src.numel()
        device = self.device

        delta_src = torch.zeros(B, device=device, dtype=torch.float32)
        delta_dst = torch.zeros(B, device=device, dtype=torch.float32)

        temp_pos: Dict[int, int] = {}
        occ_counts: Dict[int, int] = {}

        def _next_delta(u: int) -> float:
            base_pos = int(self.node_time_pos[u].item())
            offset = temp_pos.get(u, 0)
            pos = base_pos + offset

            start = int(self.node_time_indptr[u].item())
            end = int(self.node_time_indptr[u + 1].item())
            idx = start + pos

            temp_pos[u] = offset + 1
            occ_counts[u] = occ_counts.get(u, 0) + 1

            # last appearance => fallback
            if idx + 1 >= end:
                return 1.0

            t1 = self.node_time_values[idx]
            t2 = self.node_time_values[idx + 1]
            dt = (t2 - t1).float()
            dt = torch.clamp(dt, min=1.0)  # avoid 0/negative
            return float(dt.item())

        for i in range(B):
            u = int(src[i].item())
            v = int(dst[i].item())
            delta_src[i] = _next_delta(u)
            delta_dst[i] = _next_delta(v)

        return delta_src, delta_dst, occ_counts

    # ---------------------- commit after each batch ----------------------
    @torch.no_grad()
    def commit_batch(
        self,
        delta_a_nodes: Dict[int, torch.Tensor],  # {u: [K]}
        last_p_nodes: Dict[int, torch.Tensor],   # {u: [K]}
        occ_counts: Dict[int, int],              # {u: count}
        update_A_curr: bool,
    ):
        """
        Commit states AFTER optimizer.step() (or during no-grad build-null).

        - 不再更新 S_curr（不做任何 sqrt 的增量更新）
        - A_curr 是否累计由 update_A_curr 控制：
            * 训练 pass: update_A_curr=False（避免影响当前 epoch 的 null）
            * build-null pass: update_A_curr=True（从 0 开始累计 A_curr）

        Always updates:
          - p_prev
          - node_time_pos
        """

        if update_A_curr:
            if self.A_curr is None:
                raise RuntimeError("A_curr is None. Call reset_curr_from_zero() before accumulating A_curr.")
            for u, dA in delta_a_nodes.items():
                self.A_curr[int(u)] += dA.detach()

        # update p_prev
        for u, p_last in last_p_nodes.items():
            self.p_prev[int(u)] = p_last.detach()

        # advance node_time_pos
        for u, cnt in occ_counts.items():
            self.node_time_pos[int(u)] += int(cnt)


    @torch.no_grad()
    def init_old_p_cache(
        self,
        num_instances: int,
        dtype: torch.dtype = torch.float16,
        pin_memory: bool = False,
        init_uniform: bool = True,
    ):
        K = self.num_communities
        self.num_instances_cached = int(num_instances)

        if init_uniform:
            val = 1.0 / float(K)
            p_src = torch.full((num_instances, K), val, dtype=dtype, device="cpu")
            p_dst = torch.full((num_instances, K), val, dtype=dtype, device="cpu")
        else:
            p_src = torch.zeros((num_instances, K), dtype=dtype, device="cpu")
            p_dst = torch.zeros((num_instances, K), dtype=dtype, device="cpu")

        pin_ok = pin_memory and torch.cuda.is_available()
        if pin_ok:
            p_src = p_src.pin_memory()
            p_dst = p_dst.pin_memory()

        self.p_src_old_all = p_src
        self.p_dst_old_all = p_dst



    @torch.no_grad()
    def write_old_p_batch(self, start_idx: int, end_idx: int, 
                        p_src: torch.Tensor, p_dst: torch.Tensor):
        if self.p_src_old_all is None or self.p_dst_old_all is None:
            raise RuntimeError("old p cache not initialized.")
        

        p_src_copy = p_src.clone().detach().cpu().to(dtype=self.p_src_old_all.dtype)
        p_dst_copy = p_dst.clone().detach().cpu().to(dtype=self.p_dst_old_all.dtype)
        
        self.p_src_old_all[start_idx:end_idx].copy_(p_src_copy)
        self.p_dst_old_all[start_idx:end_idx].copy_(p_dst_copy)


    def read_old_p_batch(
        self,
        start_idx: int,
        end_idx: int,
        device: torch.device,
        dtype: torch.dtype,
        non_blocking: bool = True,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Called in training pass:
        Fetch old p for this batch and move to GPU (or target device).
        Returns p_src_old, p_dst_old: [B,K] on `device` with `dtype`.
        """
        if self.p_src_old_all is None or self.p_dst_old_all is None:
            raise RuntimeError("old p cache not initialized. Call init_old_p_cache(num_instances) first.")

        p_src_old = self.p_src_old_all[start_idx:end_idx].to(device=device, dtype=dtype, non_blocking=non_blocking)
        p_dst_old = self.p_dst_old_all[start_idx:end_idx].to(device=device, dtype=dtype, non_blocking=non_blocking)
        return p_src_old, p_dst_old

In [633]:
device = 'cuda' if torch.cuda.is_available() else 'mps'
node_set = set(pd.concat([link_stream['source'], link_stream['destination']], ignore_index=True))
num_nodes = len(node_set)
print(f'num_nodes = {num_nodes} ')
num_comms = 2
state_mgr = OfflineStateManager(num_nodes, num_comms, device=device)
state_mgr.pre_scan_data(link_stream)
state_mgr.init_old_p_cache(len(link_stream), dtype=torch.float16, pin_memory=True, init_uniform=True)

num_nodes = 20 
Total links: 193
T_duration = 1149
Max Lifespan: 1129.0
Total node appearances stored: 386 (expected ~ 386)


In [634]:
import math
import logging
import time
import sys
import argparse
import numpy as np
import pickle
from pathlib import Path


NUM_NEIGHBORS = 10
NUM_NEG = 1
NUM_HEADS = 2
DROP_OUT = 0.1
NUM_LAYER = 2
LEARNING_RATE = 0.01
NODE_DIM = 16
TIME_DIM = 16
USE_MEMORY = True
MESSAGE_DIM = 100
MEMORY_DIM = 16
device = 'cuda' if torch.cuda.is_available() else 'mps'

prefix = 'max_lmod'
data='link_stream'

Path("./saved_models/").mkdir(parents=True, exist_ok=True)
Path("./saved_checkpoints/").mkdir(parents=True, exist_ok=True)
MODEL_SAVE_PATH = f'./saved_models/{prefix}-{data}.pth'
get_checkpoint_path = lambda \
    epoch: f'./saved_checkpoints/{prefix}-{data}-{epoch}.pth'


logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
Path("log/").mkdir(parents=True, exist_ok=True)
fh = logging.FileHandler('log/{}.log'.format(str(time.time())))
fh.setLevel(logging.DEBUG)
ch = logging.StreamHandler()
ch.setLevel(logging.WARN)
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
fh.setFormatter(formatter)
ch.setFormatter(formatter)
logger.addHandler(fh)
logger.addHandler(ch)


In [635]:
from tgn.utils.my_data import get_data  
import importlib
import sys
importlib.reload(sys.modules['tgn.utils.my_data'])

data = 'sync_net_temp'

node_feat, edge_feat, full_data = get_data(data)
max_idx = max(full_data.unique_nodes)

cannot find (./data/sync_net_temp.npy), use zero-vector (dim=16)...
cannot find node feature: ./data/sync_net_temp_node.npy), use zero vector(dim=16)...
The dataset has 193 interactions, involving 20 different nodes


In [636]:
'''
BATCH_SIZE = 200
num_instance = len(full_data.sources)

state_mgr.reset_time_pos()
for k in range(1, 2):
    
    start_idx = BATCH_SIZE * k
    end_idx = min(num_instance, BATCH_SIZE * (k + 1))

    sources_batch = full_data.sources[start_idx:end_idx]
    destinations_batch = full_data.destinations[start_idx:end_idx]
    timestamps_batch = full_data.timestamps[start_idx:end_idx]
    edge_idxs_batch = full_data.edge_idxs[start_idx:end_idx]
    
    src = torch.as_tensor(sources_batch, dtype=torch.long, device=device)
    dst = torch.as_tensor(destinations_batch, dtype=torch.long, device=device)
    ts  = torch.as_tensor(timestamps_batch, dtype=torch.float32, device=device)
    print("src:", src)
    delta_src, delta_dst, occ_counts = state_mgr.get_delta_for_batch(src, dst)
    print("delta_src:", delta_src)
    print("occ count", occ_counts)
'''

'\nBATCH_SIZE = 200\nnum_instance = len(full_data.sources)\n\nstate_mgr.reset_time_pos()\nfor k in range(1, 2):\n\n    start_idx = BATCH_SIZE * k\n    end_idx = min(num_instance, BATCH_SIZE * (k + 1))\n\n    sources_batch = full_data.sources[start_idx:end_idx]\n    destinations_batch = full_data.destinations[start_idx:end_idx]\n    timestamps_batch = full_data.timestamps[start_idx:end_idx]\n    edge_idxs_batch = full_data.edge_idxs[start_idx:end_idx]\n\n    src = torch.as_tensor(sources_batch, dtype=torch.long, device=device)\n    dst = torch.as_tensor(destinations_batch, dtype=torch.long, device=device)\n    ts  = torch.as_tensor(timestamps_batch, dtype=torch.float32, device=device)\n    print("src:", src)\n    delta_src, delta_dst, occ_counts = state_mgr.get_delta_for_batch(src, dst)\n    print("delta_src:", delta_src)\n    print("occ count", occ_counts)\n'

In [637]:
from tgn.utils.utils import EarlyStopMonitor, get_neighbor_finder, MLP

ngh_finder = get_neighbor_finder(full_data, uniform=True, max_node_idx=max_idx)

In [638]:
from tgn.utils.my_data import get_data, compute_time_statistics, compute_time_statistics_undirected
import importlib
import sys
importlib.reload(sys.modules['tgn.utils.my_data'])

mean_time_shift, std_time_shift= compute_time_statistics_undirected(full_data.sources, 
                                                                    full_data.destinations, 
                                                                    full_data.timestamps)


In [639]:
from tgn.model.my_tgn import TGN
importlib.reload(sys.modules['tgn.model.my_tgn'])
from pathlib import Path


NUM_NEIGHBORS = 10
NUM_NEG = 1
NUM_HEADS = 2
DROP_OUT = 0.1
NUM_LAYER = 2
LEARNING_RATE = 0.01
NODE_DIM = 16
TIME_DIM = 16
USE_MEMORY = True
MESSAGE_DIM = 100
MEMORY_DIM = 16
num_communities = 2

device = 'cuda' if torch.cuda.is_available() else 'mps'
aggregator = 'last'
memory_update_at_end = False
embedding_module = 'graph_attention'
message_function = 'mlp'

prefix = 'sync_net'

tgn = TGN(
    neighbor_finder=ngh_finder,
    node_features=node_feat,
    edge_features=edge_feat,
    device=device,
    n_layers=NUM_LAYER,
    n_heads=NUM_HEADS,
    dropout=DROP_OUT,
    use_memory=USE_MEMORY,
    message_dimension=MESSAGE_DIM,
    memory_dimension=MEMORY_DIM,
    n_neighbors=NUM_NEIGHBORS,
    mean_time_shift=mean_time_shift,
    std_time_shift=std_time_shift,
    use_destination_embedding_in_message=True,
    use_source_embedding_in_message=True,

    memory_update_at_start=not memory_update_at_end,
    embedding_module_type=embedding_module,
    message_function=message_function,
    aggregator_type=aggregator, 
    num_communities=num_communities

).to(device)

optimizer = torch.optim.Adam(tgn.parameters(), lr=LEARNING_RATE)

In [640]:
import math 
import logging
import time


BATCH_SIZE = 200

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
Path("log/").mkdir(parents=True, exist_ok=True)
fh = logging.FileHandler('log/{}.log'.format(str(time.time())))
fh.setLevel(logging.DEBUG)
ch = logging.StreamHandler()
ch.setLevel(logging.WARN)
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
fh.setFormatter(formatter)
ch.setFormatter(formatter)
logger.addHandler(fh)
logger.addHandler(ch)


Path("./saved_models/").mkdir(parents=True, exist_ok=True)
Path("./saved_checkpoints/").mkdir(parents=True, exist_ok=True)
MODEL_SAVE_PATH = f'./saved_models/{prefix}-{data}.pth'
get_checkpoint_path = lambda \
    epoch: f'./saved_checkpoints/{prefix}-{data}-{epoch}.pth'


num_instance = len(full_data.sources)
num_batches = math.ceil(len(full_data.sources) / BATCH_SIZE)
logger.debug(f'num_batches: {num_batches}')

DEBUG:root:num_batches: 1


In [641]:
def build_batch_updates(src, dst,
                        p_src_curr, p_dst_curr,
                        p_src_old,  p_dst_old,
                        delta_src, delta_dst):
    delta_a_nodes = {}
    last_p_nodes = {}

    def _acc(node_ids, p_curr, p_old, deltas):
        for i in range(node_ids.numel()):
            u = int(node_ids[i].item())
            dt = deltas[i]
            if dt.item() <= 0:
                continue
            dtv = dt.to(dtype=p_curr.dtype)
            inc = (p_curr[i] - p_old[i]) * dtv 
            delta_a_nodes[u] = delta_a_nodes.get(u, 0) + inc
            last_p_nodes[u] = p_curr[i]            # p_prev 仍然写当前的

    _acc(src, p_src_curr, p_src_old, delta_src)
    _acc(dst, p_dst_curr, p_dst_old, delta_dst)
    return delta_a_nodes, last_p_nodes

def build_batch_updates_abs(src, dst, p_src, p_dst, delta_src, delta_dst):
    delta_a_nodes = {}
    last_p_nodes = {}

    def _acc(node_ids, probs, deltas):
        for i in range(node_ids.numel()):
            u = int(node_ids[i].item())
            dt = deltas[i]
            if dt.item() <= 0:
                continue
            dtv = dt.to(dtype=probs.dtype)

            inc = probs[i] * dtv          # [K]
            delta_a_nodes[u] = delta_a_nodes.get(u, 0) + inc
            last_p_nodes[u] = probs[i]


    _acc(src, p_src, delta_src)
    _acc(dst, p_dst, delta_dst)

    return delta_a_nodes, last_p_nodes

In [642]:
def print_grads(tag: str):
    print(f"\n=== Batch {k+1} {tag} Gradients ===")
    for name, param in tgn.named_parameters():
        if param.grad is not None:
            g = param.grad
            print(f"{name}: norm={g.norm().item():.6f}, max={g.abs().max().item():.6f}")


In [652]:
import importlib, sys, time
import tgn.model.loss as loss_mod
importlib.reload(loss_mod)
longitudinal_modularity_loss = loss_mod.longitudinal_modularity_loss
NUM_EPOCHS = 10


for epoch in range(NUM_EPOCHS):
    tgn = tgn.train()
    start_epoch_time = time.time()
    if USE_MEMORY:
        tgn.memory.__init_memory__()
    state_mgr.reset_time_pos()
    state_mgr.reset_p_prev_uniform()
    tgn.set_neighbor_finder(ngh_finder)
    
    epoch_loss = 0.0
    epoch_obs_loss = 0.0
    epoch_null_loss = 0.0
    epoch_csc_loss = 0.0
    epoch_balance_loss = 0.0
    epoch_conf_loss = 0.0
    epoch_steps = 0

    logger.info(f'Starting epoch {epoch+1} / {NUM_EPOCHS} ')
    for k in range(0, num_batches):
        print(f'Processing batch {k+1} / {num_batches} ')
        start_idx = BATCH_SIZE * k
        end_idx = min(num_instance, BATCH_SIZE * (k + 1))

        sources_batch = full_data.sources[start_idx:end_idx]
        destinations_batch = full_data.destinations[start_idx:end_idx]
        timestamps_batch = full_data.timestamps[start_idx:end_idx]
        edge_idxs_batch = full_data.edge_idxs[start_idx:end_idx]
        
        src = torch.as_tensor(sources_batch, dtype=torch.long, device=device)
        dst = torch.as_tensor(destinations_batch, dtype=torch.long, device=device)
        ts  = torch.as_tensor(timestamps_batch, dtype=torch.float32, device=device)

        delta_src, delta_dst, occ_counts = state_mgr.get_delta_for_batch(src, dst)

        p_src, p_dst = tgn.compute_community_prob(sources_batch, destinations_batch, timestamps_batch, edge_idxs_batch) # [B, K]
        p_src_old, p_dst_old = state_mgr.read_old_p_batch(
            start_idx, end_idx,
            device=device,
            dtype=p_src.dtype,
            non_blocking=True
        )
        
        if not torch.isfinite(p_src).all(): raise RuntimeError("p_src NaN right after forward")
        if not torch.isfinite(p_dst).all(): raise RuntimeError("p_dst NaN right after forward")

        delta_a_nodes, last_p_nodes = build_batch_updates(src, dst,
                                                          p_src, p_dst,
                                                          p_src_old, p_dst_old,
                                                          delta_src, delta_dst)

        # Compute longitudinal modularity loss
        loss, last_components, terms_raw = longitudinal_modularity_loss(
            p_src, p_dst,
            src, dst,
            delta_a_nodes,
            state_mgr.A_old, state_mgr.S_old, state_mgr.global_degree,
            state_mgr.m, state_mgr.total_duration,
            state_mgr.p_prev,
            obs_weight=1.0,
            null_weight=1.0,
            csc_weight=0.0,
            collapse_weight=3.0,
            conf_weight=0.0,
            csc_norm="l2"
        )
        obs_loss = terms_raw["obs"]
        null_loss = terms_raw["null"]


        # 1) null 项梯度
        optimizer.zero_grad(set_to_none=True)
        null_loss.backward(retain_graph=True)
        print_grads("Null Loss")

        # 2) obs 项梯度
        optimizer.zero_grad(set_to_none=True)
        obs_loss.backward(retain_graph=True)
        print_grads("Obs Loss")


        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        
        state_mgr.commit_batch(delta_a_nodes, last_p_nodes, occ_counts, update_A_curr=False)
        if USE_MEMORY:
            tgn.memory.detach_memory()
        
        epoch_loss += float(loss.detach().item())
        epoch_obs_loss  += float(last_components[0].item())
        epoch_null_loss += float(last_components[1].item())
        epoch_csc_loss  += float(last_components[2].item())
        epoch_balance_loss += float(last_components[3].item())
        epoch_conf_loss += float(last_components[4].item())
        epoch_steps += 1
        # print(f'Epoch {epoch} Batch {k+1}/{num_batches} null model loss: {epoch_null_loss / epoch_steps} ')
    if epoch_steps > 0:
        mean_loss = epoch_loss / epoch_steps
        mean_obs  = epoch_obs_loss  / epoch_steps
        mean_null = epoch_null_loss / epoch_steps
        mean_csc  = epoch_csc_loss  / epoch_steps
        mean_balance = epoch_balance_loss / epoch_steps
        mean_conf = epoch_conf_loss / epoch_steps
    
        logger.info(
            f"[Epoch {epoch}] mean loss={mean_loss:.6f} | "
            f"obs={mean_obs:.6f}, null={mean_null:.6f}, csc={mean_csc:.6f}, balance={mean_balance:.6f}, conf={mean_conf:.6f} "
        )

    # build null buffers
    tgn.eval()
    with torch.no_grad():
        if USE_MEMORY:
            tgn.memory.__init_memory__()
        state_mgr.reset_time_pos()
        state_mgr.reset_p_prev_uniform()
        state_mgr.reset_curr_from_zero()
        tgn.set_neighbor_finder(ngh_finder)
        for k in range(num_batches):
            start_idx = BATCH_SIZE * k
            end_idx = min(num_instance, BATCH_SIZE * (k + 1))

            sources_batch = full_data.sources[start_idx:end_idx]
            destinations_batch = full_data.destinations[start_idx:end_idx]
            timestamps_batch = full_data.timestamps[start_idx:end_idx]
            edge_idxs_batch = full_data.edge_idxs[start_idx:end_idx]
            
            src = torch.as_tensor(sources_batch, dtype=torch.long, device=device)
            dst = torch.as_tensor(destinations_batch, dtype=torch.long, device=device)
            ts  = torch.as_tensor(timestamps_batch, dtype=torch.float32, device=device)

            delta_src, delta_dst, occ_counts = state_mgr.get_delta_for_batch(src, dst)

            p_src_ng, p_dst_ng = tgn.compute_community_prob(sources_batch, destinations_batch, timestamps_batch, edge_idxs_batch) # [B, K]

            state_mgr.write_old_p_batch(
                start_idx, end_idx,
                p_src_ng.detach(), p_dst_ng.detach()
            )

            delta_a_nodes, last_p_nodes = build_batch_updates_abs(  
                src, dst,
                p_src_ng, p_dst_ng,
                delta_src, delta_dst)
            state_mgr.commit_batch(delta_a_nodes, last_p_nodes, occ_counts, update_A_curr=True)
            
            if USE_MEMORY:
                tgn.memory.detach_memory()
        state_mgr.finalize_curr()
        state_mgr.promote_curr_to_old(copy_A=True)
    if USE_MEMORY:
        # Backup and restore memory
        memory_backup = tgn.memory.backup_memory()
        tgn.memory.restore_memory(memory_backup)

INFO:root:Starting epoch 1 / 10 


Processing batch 1 / 1 


INFO:root:[Epoch 0] mean loss=0.989071 | obs=-0.550867, null=1.522372, csc=0.094550, balance=0.005855, conf=0.563164 



=== Batch 1 Null Loss Gradients ===
time_encoder.w.weight: norm=0.287863, max=0.146270
time_encoder.w.bias: norm=0.001536, max=0.000951
embedding_module.attention_models.0.merger.fc1.weight: norm=0.005951, max=0.001731
embedding_module.attention_models.0.merger.fc1.bias: norm=0.002384, max=0.002083
embedding_module.attention_models.0.merger.fc2.weight: norm=0.008851, max=0.002540
embedding_module.attention_models.0.merger.fc2.bias: norm=0.002239, max=0.001448
embedding_module.attention_models.0.multi_head_target.q_proj_weight: norm=0.003560, max=0.000324
embedding_module.attention_models.0.multi_head_target.k_proj_weight: norm=0.004864, max=0.000793
embedding_module.attention_models.0.multi_head_target.v_proj_weight: norm=0.005289, max=0.000926
embedding_module.attention_models.0.multi_head_target.in_proj_bias: norm=0.001308, max=0.000339
embedding_module.attention_models.0.multi_head_target.out_proj.weight: norm=0.006043, max=0.001045
embedding_module.attention_models.0.multi_head_ta

INFO:root:Starting epoch 2 / 10 


...


=== Batch 1 Null Loss Gradients ===
time_encoder.w.weight: norm=23.685843, max=13.766936
time_encoder.w.bias: norm=0.095879, max=0.064137
embedding_module.attention_models.0.merger.fc1.weight: norm=0.352150, max=0.059369
embedding_module.attention_models.0.merger.fc1.bias: norm=0.054697, max=0.024390
embedding_module.attention_models.0.merger.fc2.weight: norm=0.676289, max=0.289317
embedding_module.attention_models.0.merger.fc2.bias: norm=0.059109, max=0.030511
embedding_module.attention_models.0.multi_head_target.q_proj_weight: norm=0.051219, max=0.005451
embedding_module.attention_models.0.multi_head_target.k_proj_weight: norm=0.075780, max=0.012896
embedding_module.attention_models.0.multi_head_target.v_proj_weight: norm=0.262871, max=0.040915
embedding_module.attention_models.0.multi_head_target.in_proj_bias: norm=0.133799, max=0.041517
embedding_module.attention_models.0.multi_head_target.out_proj.weight: norm=0.279553, max=0.029359
embedding_module.attention_models.0.multi_head_

INFO:root:Starting epoch 10 / 10 


Processing batch 1 / 1 


INFO:root:[Epoch 9] mean loss=0.957366 | obs=-0.548300, null=1.367157, csc=0.131124, balance=0.046170, conf=0.530764 



=== Batch 1 Null Loss Gradients ===
time_encoder.w.weight: norm=4.986567, max=3.105858
time_encoder.w.bias: norm=0.022403, max=0.011786
embedding_module.attention_models.0.merger.fc1.weight: norm=0.072704, max=0.016603
embedding_module.attention_models.0.merger.fc1.bias: norm=0.011045, max=0.005928
embedding_module.attention_models.0.merger.fc2.weight: norm=0.206317, max=0.078164
embedding_module.attention_models.0.merger.fc2.bias: norm=0.010200, max=0.004408
embedding_module.attention_models.0.multi_head_target.q_proj_weight: norm=0.027211, max=0.002505
embedding_module.attention_models.0.multi_head_target.k_proj_weight: norm=0.041336, max=0.004698
embedding_module.attention_models.0.multi_head_target.v_proj_weight: norm=0.036179, max=0.006164
embedding_module.attention_models.0.multi_head_target.in_proj_bias: norm=0.015012, max=0.004984
embedding_module.attention_models.0.multi_head_target.out_proj.weight: norm=0.048414, max=0.005674
embedding_module.attention_models.0.multi_head_ta

In [644]:
print(state_mgr.node_lifespans)

tensor([ 945.,  721.,  961.,  946.,  910.,  957., 1026., 1069., 1082., 1049.,
         932., 1003., 1073.,  969., 1009.,  918.,  820., 1119., 1129.,  949.],
       device='mps:0')


In [645]:
print(state_mgr.global_degree)

tensor([18., 18., 16., 16., 19., 16., 15., 21., 21., 28., 25., 14., 23., 19.,
        21., 17., 22., 20., 29.,  8.], device='mps:0')


In [653]:
print(state_mgr.A_old)

tensor([[735.1859, 210.8141],
        [442.3968, 279.6033],
        [666.9681, 295.0318],
        [578.8881, 368.1119],
        [597.3635, 313.6365],
        [705.5152, 252.4847],
        [613.3346, 413.6653],
        [682.1613, 387.8387],
        [730.2656, 352.7344],
        [630.5846, 419.4153],
        [564.8522, 368.1478],
        [599.5167, 404.4833],
        [640.7053, 433.2946],
        [550.6177, 419.3823],
        [578.9155, 431.0844],
        [614.0775, 304.9225],
        [527.0897, 293.9103],
        [666.2693, 453.7307],
        [688.5992, 441.4008],
        [625.1055, 324.8946]], device='mps:0')


In [647]:
print(state_mgr.S_old.sum())

tensor(17071.6445, device='mps:0')


In [648]:
print((state_mgr.S_old.sum())**2/(2*state_mgr.m*state_mgr.total_duration))

tensor(657.1180, device='mps:0')


In [649]:
print(state_mgr.m)

193.0


In [650]:
print(state_mgr.total_duration)

1149.0


In [654]:
import json
import numpy as np
import pandas as pd
import torch

def export_probs_to_csv(
    tgn,
    full_data,
    BATCH_SIZE: int,
    device,
    out_csv_path: str,
):
    tgn.eval()

    num_instance = len(full_data.sources)
    rows = []

    with torch.no_grad():
        for k in range((num_instance + BATCH_SIZE - 1) // BATCH_SIZE):
            start_idx = BATCH_SIZE * k
            end_idx = min(num_instance, BATCH_SIZE * (k + 1))

            sources_batch = full_data.sources[start_idx:end_idx]
            destinations_batch = full_data.destinations[start_idx:end_idx]
            timestamps_batch = full_data.timestamps[start_idx:end_idx]
            edge_idxs_batch = full_data.edge_idxs[start_idx:end_idx]

            # p_src/p_dst: torch.Tensor [B, K]
            p_src, p_dst = tgn.compute_community_prob(
                sources_batch, destinations_batch, timestamps_batch, edge_idxs_batch
            )

            # move to CPU numpy
            p_src_np = p_src.detach().to("cpu").float().numpy()
            p_dst_np = p_dst.detach().to("cpu").float().numpy()

            # argmax hard assignment
            src_commu = p_src_np.argmax(axis=1).astype(np.int64)
            dst_commu = p_dst_np.argmax(axis=1).astype(np.int64)

            # build rows
            for i in range(end_idx - start_idx):
                rows.append({
                    "source": int(sources_batch[i]),
                    "destination": int(destinations_batch[i]),
                    "timestamp": float(timestamps_batch[i]),
                    "p_src": json.dumps(p_src_np[i].tolist()),  # store vector as JSON string
                    "p_dst": json.dumps(p_dst_np[i].tolist()),
                    "src_commu": int(src_commu[i]),
                    "dst_commu": int(dst_commu[i]),
                })

    df = pd.DataFrame(rows, columns=[
        "source", "destination", "timestamp",
        "p_src", "p_dst", "src_commu", "dst_commu"
    ])
    df.to_csv(out_csv_path, index=False)
    print(f"Saved: {out_csv_path}  (rows={len(df)})")


# 用法示例：
export_probs_to_csv(tgn, full_data, BATCH_SIZE, device, "pred_probs.csv")

Saved: pred_probs.csv  (rows=193)
